#### Enviroment Setup

In [1]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Loading dataset and filtering it

In [2]:
import pandas as pd
from bertopic import BERTopic

C:\Users\emman\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Reading csv, extracting useful columns, appending future columns

df = pd.read_csv('dataset/datasetA.csv')
df = df[['title', 'link', 'source', 'classes_str']]
df[['text', 'authors', 'keywords', 'summary']] = ''

In [ ]:
#Selecting just the title and classes_str columns. Put in a new dataframe so its less confusing for the rest of the code
titles_and_classes = df[['title', 'classes_str']]
#Checking nulls 
titles_and_classes.isna().sum()

In [ ]:
#Filtering the df so it only selects rows where the classes_str value contains 'Learning, Knowledge & Education'
df2 = titles_and_classes[titles_and_classes['classes_str'].str.contains('Learning, Knowledge & Education', case=False)]

#### Testing Various Topic Modeling Techniques

In [ ]:
#BERTopic only reads lists so i did this
docs = df2["title"].tolist()
print(len(docs))

In [ ]:
#Default BERTopic Model 
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()
"""
Topic is the topic number. -1 refers to outliers so it can be ignored.
Count is how many rows from title fit that topic
Name is the topic name 
Representation is the top words that summarize the topic
Representative_Docs is the example documents that best illustrate the topic"""

In [ ]:
#Work in progress
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
topic_model = BERTopic(embedding_model=embedding_model)
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()

In [ ]:
#Work in progress
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model)
topics, probs = topic_model.fit_transform(docs)
topics = topic_model.reduce_topics(docs, nr_topics=10)  
topic_model.get_topic_info()

### Webscraping

In [5]:
from newspaper import Article
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\emman\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# log file for Emmanuel's debuging ventures
import logging
logging.basicConfig(filename='scraper.log', format='%(levelname)s: %(asctime)s: %(message)s', datefmt='%m/%d/%Y - %I:%M:%S %p', level=logging.DEBUG)
logging.getLogger('selenium').setLevel(logging.WARNING)
log = logging.getLogger(__name__)
#levels: debug, info, warning, error, critical

In [6]:
# all selenium webdriver setting for max efficiency
def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-renderer-backgrounding")
    chrome_options.add_argument("--disable-background-timer-throttling")
    chrome_options.add_argument("--disable-backgrounding-occluded-windows")
    chrome_options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(chrome_options)
    return driver

In [20]:
# parses the javascript of the html to redirect and retrieve the correct url
def retrieve_url(driver, prefetch_url):
    driver.get(prefetch_url)
    url = driver.current_url
    while url.startswith('https://news.google.com/rss/articles/CBMi'):
        url = driver.current_url
    return url

In [8]:
# return dictionary of authors, keywords, summary, and text
def scrape(url):
    article = Article(url)
    article.build()
    return {'authors': article.authors,
            'keywords': article.keywords,
            'summary': article.summary,
            'text': article.text}

def scrape_text(url):
    article = Article(url)
    article.download()
    article.parse()
    return article.text